In [1]:
library(DESeq2)
library(sva)
library(STRINGdb)
library(org.Mm.eg.db)

top_dir =  '/Users/jcasalet/Desktop/RESEARCH/LIVER/DATA/HI/4_CRISP/'
output_dir = paste(top_dir, "NORMALIZE_THEN_MERGE", sep="/")
meta <- read.csv(paste(top_dir, 'metadata.csv', sep="/"), header=TRUE, row.names=1, stringsAsFactors=TRUE)

Loading required package: S4Vectors

Loading required package: stats4

Loading required package: BiocGenerics


Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:base’:

    anyDuplicated, append, as.data.frame, basename, cbind, colnames,
    dirname, do.call, duplicated, eval, evalq, Filter, Find, get, grep,
    grepl, intersect, is.unsorted, lapply, Map, mapply, match, mget,
    order, paste, pmax, pmax.int, pmin, pmin.int, Position, rank,
    rbind, Reduce, rownames, sapply, setdiff, sort, table, tapply,
    union, unique, unsplit, which.max, which.min



Attaching package: ‘S4Vectors’


The following objects are masked from ‘package:base’:

    expand.grid, I, unname


Loading required package: IRanges

Loading required package: GenomicRanges

Loading required package: GenomeInfoDb

Loading required package: SummarizedExperiment

Loading required package: MatrixGe

In [2]:
getCounts <- function(expr_file, meta_file, norm){
    expr <- read.csv(paste(top_dir, expr_file, sep="/"), header=TRUE, row.names=1, stringsAsFactors=TRUE, check.names=FALSE)
    meta <- read.csv(paste(top_dir, meta_file, sep="/"), header=TRUE, row.names=1, stringsAsFactors=TRUE)
    expr_final <- expr[rownames(meta)]
    dds <- DESeqDataSetFromMatrix(ceiling(expr_final), meta['dataset'], ~1)
    dds_sf <- estimateSizeFactors(dds)
    return(counts(dds_sf, normalized=norm))
}

In [3]:
norm_47_complete <- getCounts('glds_47_complete.csv', 'meta-47_permuted.csv', TRUE)
norm_48_complete <- getCounts('glds_48_complete.csv', 'meta-48_permuted.csv', TRUE)
norm_137_complete <- getCounts('glds_137_complete.csv', 'meta-137_permuted.csv', TRUE)
norm_168_complete <- getCounts('glds_168_complete.csv', 'meta-168_permuted.csv', TRUE)

norm_47_no_pseudogenes <- getCounts('glds_47_no_pseudogenes.csv', 'meta-47_permuted.csv', TRUE)
norm_48_no_pseudogenes <- getCounts('glds_48_no_pseudogenes.csv', 'meta-48_permuted.csv', TRUE)
norm_137_no_pseudogenes <- getCounts('glds_137_no_pseudogenes.csv', 'meta-137_permuted.csv', TRUE)
norm_168_no_pseudogenes <- getCounts('glds_168_no_pseudogenes.csv', 'meta-168_permuted.csv', TRUE)

norm_47_no_pseudogenes_wExternal <- getCounts('glds_47_no_pseudogenes_wExternal.csv', 'meta-47_permuted.csv', TRUE)
norm_48_no_pseudogenes_wExternal <- getCounts('glds_48_no_pseudogenes_wExternal.csv', 'meta-48_permuted.csv', TRUE)
norm_137_no_pseudogenes_wExternal <- getCounts('glds_137_no_pseudogenes_wExternal.csv', 'meta-137_permuted.csv', TRUE)
norm_168_no_pseudogenes_wExternal <- getCounts('glds_168_no_pseudogenes_wExternal.csv', 'meta-168_permuted.csv', TRUE)



converting counts to integer mode

converting counts to integer mode

converting counts to integer mode

converting counts to integer mode

converting counts to integer mode

converting counts to integer mode

converting counts to integer mode

converting counts to integer mode

converting counts to integer mode

converting counts to integer mode

converting counts to integer mode

converting counts to integer mode



In [4]:
unnorm_47_complete <- getCounts('glds_47_complete.csv', 'meta-47_permuted.csv', FALSE)
unnorm_48_complete <- getCounts('glds_48_complete.csv', 'meta-48_permuted.csv', FALSE)
unnorm_137_complete <- getCounts('glds_137_complete.csv', 'meta-137_permuted.csv', FALSE)
unnorm_168_complete <- getCounts('glds_168_complete.csv', 'meta-168_permuted.csv', FALSE)

unnorm_47_no_pseudogenes <- getCounts('glds_47_no_pseudogenes.csv', 'meta-47_permuted.csv', FALSE)
unnorm_48_no_pseudogenes <- getCounts('glds_48_no_pseudogenes.csv', 'meta-48_permuted.csv', FALSE)
unnorm_137_no_pseudogenes <- getCounts('glds_137_no_pseudogenes.csv', 'meta-137_permuted.csv', FALSE)
unnorm_168_no_pseudogenes <- getCounts('glds_168_no_pseudogenes.csv', 'meta-168_permuted.csv', FALSE)

unnorm_47_no_pseudogenes_wExternal <- getCounts('glds_47_no_pseudogenes_wExternal.csv', 'meta-47_permuted.csv', FALSE)
unnorm_48_no_pseudogenes_wExternal <- getCounts('glds_48_no_pseudogenes_wExternal.csv', 'meta-48_permuted.csv', FALSE)
unnorm_137_no_pseudogenes_wExternal <- getCounts('glds_137_no_pseudogenes_wExternal.csv', 'meta-137_permuted.csv', FALSE)
unnorm_168_no_pseudogenes_wExternal <- getCounts('glds_168_no_pseudogenes_wExternal.csv', 'meta-168_permuted.csv', FALSE)



converting counts to integer mode

converting counts to integer mode

converting counts to integer mode

converting counts to integer mode

converting counts to integer mode

converting counts to integer mode

converting counts to integer mode

converting counts to integer mode

converting counts to integer mode

converting counts to integer mode

converting counts to integer mode

converting counts to integer mode



In [5]:
# Function to merge 2 dfs on row names and remove non-common row names
uniqMerge <- function(df1, df2, df3, df4){
    merged_1 <- merge(df1, df2, by=0, all=FALSE, no.dups=TRUE) # do the merge
    row.names(merged_1) = merged_1$Row.names # assign row names using the "Row.names" column
    merged_1 <- within(merged_1, rm('Row.names')) # remove the "Row.names" column
    
    merged_2 <- merge(df3, df4, by=0, all=FALSE, no.dups=TRUE) # do the merge
    row.names(merged_2) = merged_2$Row.names # assign row names using the "Row.names" column
    merged_2 <- within(merged_2, rm('Row.names')) # remove the "Row.names" column
    
    merged <- merge(merged_1, merged_2, by=0, all=FALSE, no.dups=TRUE) # do the merge
    row.names(merged) = merged$Row.names # assign row names using the "Row.names" column
    merged <- within(merged, rm('Row.names')) # remove the "Row.names" column
    
    return(merged)
    
}

In [6]:
norm_merged_complete <- uniqMerge(norm_47_complete, norm_48_complete, norm_137_complete, norm_168_complete)

norm_merged_no_pseudogenes <- uniqMerge(norm_47_no_pseudogenes,norm_48_no_pseudogenes,norm_137_no_pseudogenes, norm_168_no_pseudogenes)

norm_merged_no_pseudogenes_wExternal <- uniqMerge(norm_47_no_pseudogenes_wExternal, norm_48_no_pseudogenes_wExternal,norm_137_no_pseudogenes_wExternal,norm_168_no_pseudogenes_wExternal )


In [7]:
unnorm_merged_complete <- uniqMerge(unnorm_47_complete, unnorm_48_complete,unnorm_137_complete, unnorm_168_complete)

unnorm_merged_no_pseudogenes <- uniqMerge(unnorm_47_no_pseudogenes, unnorm_48_no_pseudogenes,unnorm_137_no_pseudogenes,  unnorm_168_no_pseudogenes)

unnorm_merged_no_pseudogenes_wExternal <- uniqMerge(unnorm_47_no_pseudogenes_wExternal, unnorm_48_no_pseudogenes_wExternal, unnorm_137_no_pseudogenes_wExternal, unnorm_168_no_pseudogenes_wExternal)


## Batch effect correction 
Since the library prep batch effect is still fairly pronounced after sequencing depth correction, we can correct the data using Combat-Seq (paper: https://pubmed.ncbi.nlm.nih.gov/33015620/)

In [8]:
norm_merged_complete=as.matrix(norm_merged_complete)
norm_merged_no_pseudogenes = as.matrix(norm_merged_no_pseudogenes)
norm_merged_no_pseudogenes_wExternal = as.matrix(norm_merged_no_pseudogenes_wExternal)

norm_merged_complete_Combat <- ComBat_seq(norm_merged_complete, batch=meta$Library.prep)
norm_merged_no_pseudogenes_Combat <- ComBat_seq(norm_merged_no_pseudogenes, batch=meta$Library.prep)
norm_merged_no_pseudogenes_wExternal_Combat <- ComBat_seq(norm_merged_no_pseudogenes_wExternal, batch=meta$Library.prep)


Found 2 batches
Using null model in ComBat-seq.
Adjusting for 0 covariate(s) or covariate level(s)
Estimating dispersions
Fitting the GLM model
Shrinkage off - using GLM estimates for parameters
Adjusting the data
Found 2 batches
Using null model in ComBat-seq.
Adjusting for 0 covariate(s) or covariate level(s)
Estimating dispersions
Fitting the GLM model
Shrinkage off - using GLM estimates for parameters
Adjusting the data
Found 2 batches
Using null model in ComBat-seq.
Adjusting for 0 covariate(s) or covariate level(s)
Estimating dispersions
Fitting the GLM model
Shrinkage off - using GLM estimates for parameters
Adjusting the data


In [9]:
unnorm_merged_complete=as.matrix(unnorm_merged_complete)
unnorm_merged_no_pseudogenes = as.matrix(unnorm_merged_no_pseudogenes)
unnorm_merged_no_pseudogenes_wExternal = as.matrix(unnorm_merged_no_pseudogenes_wExternal)

unnorm_merged_complete_Combat <- ComBat_seq(unnorm_merged_complete, batch=meta$dataset)
unnorm_merged_no_pseudogenes_Combat <- ComBat_seq(unnorm_merged_no_pseudogenes, batch=meta$dataset)
unnorm_merged_no_pseudogenes_wExternal_Combat <- ComBat_seq(unnorm_merged_no_pseudogenes_wExternal, batch=meta$dataset)

Found 4 batches
Using null model in ComBat-seq.
Adjusting for 0 covariate(s) or covariate level(s)
Estimating dispersions
Fitting the GLM model
Shrinkage off - using GLM estimates for parameters
Adjusting the data
Found 4 batches
Using null model in ComBat-seq.
Adjusting for 0 covariate(s) or covariate level(s)
Estimating dispersions
Fitting the GLM model
Shrinkage off - using GLM estimates for parameters
Adjusting the data
Found 4 batches
Using null model in ComBat-seq.
Adjusting for 0 covariate(s) or covariate level(s)
Estimating dispersions
Fitting the GLM model
Shrinkage off - using GLM estimates for parameters
Adjusting the data


## Convert to gene symbols
Map ENSEMBL gene IDs to gene symbols to make the results easier to interpret biologically. Write out these data.

In [10]:
# get ENSEMBL:symbol mapping from org.Mm.eg.db database
# drop any genes that don't have a gene symbol
mapped <- na.omit(as.data.frame(mapIds(org.Mm.eg.db, keys=rownames(norm_merged_complete),
                         keytype='ENSEMBL', column='SYMBOL', multiVals='first')))
colnames(mapped) <- 'symbol'

'select()' returned 1:many mapping between keys and columns



In [11]:
convertAndSave <- function(df, file_name, mapped) {
    df_merged <- merge(df, mapped, by=0, all=FALSE, no.dups=FALSE) # merge gene symbols into expression df
    .rowNamesDF(df_merged, make.names=TRUE) <- df_merged$symbol # make gene symbols row names
    df_merged <- within(df_merged, rm(Row.names)) # remove residual columns
    df_merged <- within(df_merged, rm(symbol))
    df_merged <- log2(df_merged+1)
    write.csv(df_merged, paste(output_dir, file_name, sep="/"), row.names=TRUE, quote=FALSE)
}

In [12]:
convertAndSave(norm_merged_complete_Combat, 'normalized_merged_corrected_log_complete.csv', mapped)
convertAndSave(norm_merged_no_pseudogenes_Combat, 'normalized_merged_corrected_log_no_pseudogenes.csv', mapped)
convertAndSave(norm_merged_no_pseudogenes_wExternal_Combat, 'normalized_merged_corrected_log_no_pseudogenes_wExternal.csv', mapped)

convertAndSave(unnorm_merged_complete_Combat, 'unnormalized_merged_corrected_log_complete.csv', mapped)
convertAndSave(unnorm_merged_no_pseudogenes_Combat, 'unnormalized_merged_corrected_log_no_pseudogenes.csv', mapped)
convertAndSave(unnorm_merged_no_pseudogenes_wExternal_Combat, 'unnormalized_merged_corrected_log_no_pseudogenes_wExternal.csv', mapped)

convertAndSave(norm_merged_complete, 'normalized_merged_log_complete.csv', mapped)
convertAndSave(norm_merged_no_pseudogenes, 'normalized_merged_log_no_pseudogenes.csv', mapped)
convertAndSave(norm_merged_no_pseudogenes_wExternal, 'normalized_merged_log_no_pseudogenes_wExternal.csv', mapped)

convertAndSave(unnorm_merged_complete, 'unnormalized_merged_log_complete.csv', mapped)
convertAndSave(unnorm_merged_no_pseudogenes, 'unnormalized_merged_log_no_pseudogenes.csv', mapped)
convertAndSave(unnorm_merged_no_pseudogenes_wExternal, 'unnormalized_merged_log_no_pseudogenes_wExternal.csv', mapped)

